In [ ]:
from pandas.core.interchange.dataframe_protocol import DataFrame

from _bootstrap import setup

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from enum import StrEnum
import os
import pandas as pd

In [ ]:
class OrientacionCorte(StrEnum):
    VERTICAL = "vertical"
    HORIZONTAL = "horizontal"

In [ ]:
def obtener_o_crear_directorio(ruta_base: Path, carpeta_destino: str):
    ruta_completa = ruta_base / carpeta_destino

    ruta_completa.mkdir(parents=True, exist_ok=True)

    if not ruta_completa.exists():
        raise RuntimeError("No se pudo crear el directorio")

    return ruta_completa


# Funciones
def obtener_nombre_carpeta_clase(valor_clase: str | int | float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'clase01', 'clase02', ... 'clase11', etc.
    - Si está vacío/NaN → 'sin clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"



In [ ]:
def obtener_rutas_perfiles(ruta_origen: Path, clase: int | str, extension: str) -> list[Path]:
    nombre_carpeta = obtener_nombre_carpeta_clase(clase)
    carpeta = ruta_origen / nombre_carpeta

    return list(carpeta.glob(f"*.{extension}"))

In [ ]:
def determinar_orientacion_corte(clase: str) -> OrientacionCorte:
    if clase in range(1, 10):
        return OrientacionCorte.VERTICAL
    return OrientacionCorte.HORIZONTAL


def crear_ruta_archivo(ruta_base: Path, nombre_archivo: str, sufijo: str =""):
    return os.path.join(ruta_base, f"{nombre_archivo}{sufijo}.png")

In [ ]:
def mostrar_imagen(imagen):
    plt.imshow(imagen, cmap=plt.cm.gray)
    plt.axis("on")
    plt.figure()


def obtener_nombre_archivo(ruta: Path) -> str:
    return Path(ruta).stem


def guardar_imagen(ruta: Path, imagen):
    cv2.imwrite(ruta, imagen)


def leer_imagen(ruta_imagen: str):
    img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    return img


def cuadrar_imagen(img):
    alto, ancho = img.shape
    lado = max(alto, ancho)

    offset_y = (lado - alto) // 2
    offset_x = (lado - ancho) // 2

    lienzo = crear_lienzo_cuadrado(img, lado)
    lienzo[offset_y:offset_y + alto, offset_x:offset_x + ancho] = img
    return lienzo


def obtener_cortes_fuente(alto: int, ancho: int, orientacion_corte: OrientacionCorte, ) -> \
        tuple[tuple[slice, slice], tuple[slice, slice]]:
    if orientacion_corte == OrientacionCorte.VERTICAL:
        return (
            (slice(0, alto // 2), slice(0, ancho)),
            (slice(alto // 2, alto), slice(0, ancho)),
        )

    return (
        (slice(0, alto), slice(0, ancho // 2)),
        (slice(0, alto), slice(ancho // 2, ancho)),
    )


def crear_lienzo_cuadrado(img, lado: int):
    return np.full((lado, lado), 255, dtype=img.dtype)


def trasladar_corte_a_lienzo(
        corte_src: tuple[slice, slice],
        offset_y: int,
        offset_x: int,
        lado_lienzo: int,
) -> tuple[slice, slice]:
    slice_y, slice_x = corte_src

    return (
        trasladar_slice(slice_y, offset_y, lado_lienzo),
        trasladar_slice(slice_x, offset_x, lado_lienzo),
    )


def trasladar_slice(segmento: slice, offset: int, limite: int) -> slice:
    inicio = offset if segmento.start is None else segmento.start + offset
    fin = limite if segmento.stop is None else segmento.stop + offset
    return slice(inicio, fin)

In [ ]:
def procesar_perfil(ruta_perfil: Path, ruta_destino_clase: Path) -> pd.DataFrame:
    try:
        img = leer_imagen(ruta_perfil)
        img_cuadrada = cuadrar_imagen(img)
        nombre_original = obtener_nombre_archivo(ruta_perfil)
        ruta_imagen_cuadrada = crear_ruta_archivo(ruta_destino_clase, nombre_original)
        print(ruta_imagen_cuadrada)
        guardar_imagen(ruta=ruta_imagen_cuadrada, imagen=img_cuadrada)

    except Exception as e:
        print(f"Error procesando {ruta_perfil}: {e}")


def procesar_clase(ruta_origen: Path, ruta_destino: Path, clase: int | str) -> None:
    perfiles = obtener_rutas_perfiles(ruta_origen, clase, extension="png")

    if not perfiles:
        return None

    ruta_destino_clase = obtener_o_crear_directorio(
        ruta_destino,
        obtener_nombre_carpeta_clase(clase)
    )
    for ruta_perfil in perfiles:
        procesar_perfil(ruta_perfil, ruta_destino_clase)
    return None


def procesar_imagenes(ruta_origen: Path, ruta_destino: Path, clases: list[int | str]) ->None:
    for clase in clases:
        procesar_clase(ruta_origen, ruta_destino, clase)


In [ ]:
from _bootstrap import setup

clases_a_cuadrar = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 'sin_clase']
PATHS = setup()

procesar_imagenes(PATHS.DATA_CLASIFICADOS, PATHS.DATA_CUADRADOS, clases_a_cuadrar)

csv_origen: Path = PATHS.DATA_CLASIFICADOS / "metadata.csv"
csv_destino: Path = PATHS.DATA_CUADRADOS / "metadata.csv"

# copiar CSV original a destino
df_completos = pd.read_csv(csv_origen)
df_completos.to_csv(csv_destino, index=False)